In [0]:
orders_df = spark.table(
    "logistics.silver.shipment_orders"
)

tracking_df = spark.table(
    "logistics.silver.shipment_tracking_events"
)

orders_df.printSchema()
tracking_df.printSchema()

In [0]:
display(
    orders_df.groupBy("shipment_status")
    .count()
    .orderBy("shipment_status")
)

In [0]:
display(
    tracking_df.groupBy("event_type")
    .count()
    .orderBy("event_type")
)

In [0]:
display(
    tracking_df.select(
        "shipment_id",
        "tracking_event_id",
        "event_type",
        "event_ts",
        "received_ts"
    ).orderBy("shipment_id", "event_ts")
)

In [0]:
from pyspark.sql.functions import col

invalid_events = tracking_df.filter(
    col("shipment_id").isNull() |
    col("tracking_event_id").isNull() |
    col("event_ts").isNull() |
    col("event_type").isNull()
)

valid_events = tracking_df.filter(
    col("shipment_id").isNotNull() &
    col("tracking_event_id").isNotNull() &
    col("event_ts").isNotNull() &
    col("event_type").isNotNull()
)

print("Valid events:", valid_events.count())
print("Invalid events:", invalid_events.count())

In [0]:
allowed_events = [
    "PICKED_UP",
    "AT_HUB",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

unknown_events = valid_events.filter(
    ~col("event_type").isin(allowed_events)
)

display(unknown_events)

In [0]:
display(
    valid_events.groupBy("tracking_event_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dedup_window = Window.partitionBy(
    "tracking_event_id"
).orderBy(
    col("received_ts").asc_nulls_last()
)

dedup_events = (
    valid_events
    .withColumn(
        "rn",
        row_number().over(dedup_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
from pyspark.sql.functions import (
    col, row_number
)
from pyspark.sql.window import Window

history_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").asc(),
    col("received_ts").asc_nulls_last(),
    col("tracking_event_id").asc()
)

status_history = dedup_events.withColumn(
    "event_sequence",
    row_number().over(history_window)
)

In [0]:
display(
    status_history.select(
        "shipment_id",
        "tracking_event_id",
        "event_type",
        "event_ts",
        "received_ts",
        "event_sequence"
    ).orderBy("shipment_id", "event_sequence")
)

In [0]:
current_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").desc(),
    col("received_ts").desc_nulls_last(),
    col("tracking_event_id").desc()
)

current_status = (
    status_history
    .withColumn(
        "rn",
        row_number().over(current_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
display(
    current_status.select(
        "shipment_id",
        "event_type",
        "event_ts",
        "received_ts"
    )
)

In [0]:
comparison_df = (
    current_status.alias("t")
    .join(
        orders_df.alias("o"),
        on="shipment_id",
        how="left"
    )
    .select(
        col("shipment_id"),
        col("event_type").alias("tracking_status"),
        col("shipment_status").alias("order_status"),
        col("event_ts")
    )
)

display(comparison_df)

In [0]:
status_history.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_status_history"
    )

In [0]:
current_status.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_current"
    )

In [0]:
display(
    current_status.groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
display(
    current_status.groupBy("event_type")
    .count()
    .orderBy("event_type")
)

In [0]:
print("History records:", status_history.count())

print(
    "Current shipments:",
    current_status.select("shipment_id").distinct().count()
)